# Diabetic Retinopathy Classification - No Transformations + CUDA Version

Multi-class CNN classifier for diabetic retinopathy stages (0–4).

## Key Differences from Main Notebook

This notebook is a **baseline version** that trains on raw images **without any preprocessing or augmentation**:

### What's Removed
- ❌ No retinal enhancement (disk cropping, CLAHE, vessel masks)
- ❌ No data augmentation (no flips, rotations, color jitter, gaussian blur, random erasing)
- ❌ No random cropping

### What's Used
- ✅ Raw fundus images from `data/` folder
- ✅ Simple resize to 320×320
- ✅ ImageNet normalization only
- ✅ **Direct GPU/CUDA memory loading** for maximum speed (all images pre-loaded as GPU tensors)

### Key Optimization: Direct GPU Loading
All images are loaded directly into CUDA/GPU memory during initialization. This eliminates **both**:
1. Disk I/O during training (already in memory)
2. CPU→GPU transfers during training (already on GPU)

**Trade-off**: Requires ~5-6 GB of GPU VRAM but provides the fastest possible training speed.

### Purpose
This notebook serves as a **baseline** to compare against the full preprocessing pipeline. It answers the question: "How much do transformations actually help?"

---

**Dataset**: `data/` (1216 images, 5 classes) — random 70/15/15 stratified split

**Architecture**: EfficientNetB0 (pretrained ImageNet) with custom classifier head

Classes: `0-noDR` | `1-mild` | `2-moderate` | `3-severe` | `4-proliferativeDR`

In [1]:
import os
# os.chdir('/Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3')
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3


In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import random, os, time
import numpy as np
from PIL import Image, ImageOps
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'Using device: {device}')

DATA_DIR  = 'data'
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

CLASS_NAMES   = ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']
NUM_CLASSES   = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

Using device: mps
Classes (5): ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']


## Dataset - Direct GPU Loading

The `data/` directory contains **1216 raw fundus images** in 5 class folders (not pre-split).
We apply a random 70/15/15 stratified split into train/val/test.

**Key optimization**: All images are **pre-loaded directly into GPU memory** during dataset initialization. This eliminates:
- ✅ Disk I/O during training (images cached in memory)
- ✅ CPU→GPU transfers during training (tensors already on GPU)

| Class | Count |
|---|---|
| 0-noDR | 467 |
| 1-mild | 213 |
| 2-moderate | 354 |
| 3-severe | 95 |
| 4-proliferativeDR | 87 |
| **Total** | **1216** |

The dataset is **highly imbalanced**. We use inverse-frequency class weights in the loss function to handle this.

**GPU memory usage**: ~5-6 GB for all images as GPU tensors (320×320×3 float32)

In [ ]:
class FlatImageDataset(Dataset):
    """
    Loads images from root/class_name/image.jpg flat-folder structure.
    Supports an optional index subset for train/val/test splitting.
    """
    def __init__(self, root_dir, indices=None, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        class_names = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = int(class_name.split('-')[0])
            for fname in sorted(os.listdir(class_dir)):
                # Skip macOS system files
                if fname.startswith('._'):
                    continue
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, fname))
                    self.labels.append(label)
        if indices is not None:
            self.image_paths = [self.image_paths[i] for i in indices]
            self.labels      = [self.labels[i]      for i in indices]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with Image.open(self.image_paths[idx]) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, self.labels[idx]


class GPUImageDataset(Dataset):
    """
    Pre-loads all images directly into GPU memory for maximum training speed.
    Loads images once during initialization and stores them as GPU tensors.
    This eliminates both disk I/O and CPU→GPU transfers during training.
    """
    def __init__(self, root_dir, indices=None, transform=None, device='cuda'):
        """
        Args:
            root_dir: root directory with class subdirectories
            indices: optional list of indices to use (for train/val/test split)
            transform: base transform to apply once during loading (resize, normalize)
            device: device to load tensors onto ('cuda', 'mps', or 'cpu')
        """
        print(f'Loading images directly into {device} memory from {root_dir}...')
        self.device = device
        
        # Collect image paths and labels
        image_paths = []
        labels = []
        class_names = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = int(class_name.split('-')[0])
            for fname in sorted(os.listdir(class_dir)):
                # Skip macOS system files
                if fname.startswith('._'):
                    continue
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(class_dir, fname))
                    labels.append(label)
        
        # Apply index subset if provided
        if indices is not None:
            image_paths = [image_paths[i] for i in indices]
            labels = [labels[i] for i in indices]
        
        # Pre-load all images directly into GPU memory
        self.images = []
        self.labels = torch.tensor(labels, dtype=torch.long, device=device)
        start_time = time.time()
        
        for idx, path in enumerate(image_paths):
            with Image.open(path) as img:
                img = img.convert('RGB')
                if transform:
                    img = transform(img)
                # Move tensor to GPU immediately after transformation
                if torch.is_tensor(img):
                    img = img.to(device)
                self.images.append(img)
            
            if (idx + 1) % 200 == 0:
                elapsed = time.time() - start_time
                print(f'  Loaded {idx+1}/{len(image_paths)} images onto {device} ({elapsed:.1f}s)')
        
        elapsed = time.time() - start_time
        print(f'  Loaded all {len(image_paths)} images onto {device} in {elapsed:.1f}s')
        
        # Calculate GPU memory usage
        if len(self.images) > 0 and torch.is_tensor(self.images[0]):
            mem_per_image = self.images[0].element_size() * self.images[0].nelement()
            total_mem_mb = (mem_per_image * len(self.images)) / (1024 * 1024)
            print(f'  GPU memory usage: {total_mem_mb:.1f} MB')
            
            # Show current GPU memory stats if using CUDA
            if device.startswith('cuda') and torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated(device) / (1024**3)
                reserved = torch.cuda.memory_reserved(device) / (1024**3)
                print(f'  CUDA memory allocated: {allocated:.2f} GB, reserved: {reserved:.2f} GB')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Images and labels are already on GPU, no transfer needed
        return self.images[idx], self.labels[idx]


def stratified_split(root_dir, train_frac=0.70, val_frac=0.15, seed=42):
    """Return (train_indices, val_indices, test_indices) via per-class stratified split."""
    rng = random.Random(seed)
    class_names = sorted([d for d in os.listdir(root_dir)
                           if os.path.isdir(os.path.join(root_dir, d))])
    all_labels = []
    idx_counter = 0
    class_to_indices = {}
    for class_name in class_names:
        class_dir = os.path.join(root_dir, class_name)
        label = int(class_name.split('-')[0])
        idxs = []
        for fname in sorted(os.listdir(class_dir)):
            # Skip macOS system files
            if fname.startswith('._'):
                continue
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                idxs.append(idx_counter)
                all_labels.append(label)
                idx_counter += 1
        class_to_indices[label] = idxs

    train_idx, val_idx, test_idx = [], [], []
    for label, idxs in class_to_indices.items():
        shuffled = idxs[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(n * train_frac))
        n_val   = max(1, int(n * val_frac))
        train_idx.extend(shuffled[:n_train])
        val_idx.extend(  shuffled[n_train:n_train + n_val])
        test_idx.extend( shuffled[n_train + n_val:])

    return train_idx, val_idx, test_idx


# Base transforms: just resize and normalize (applied once during loading)
base_transform = transforms.Compose([
    transforms.Resize((320, 320),
                      interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Stratified 70/15/15 split
train_idx, val_idx, test_idx = stratified_split(DATA_DIR, train_frac=0.70, val_frac=0.15, seed=SEED)

# Load all images directly into GPU memory
train_dataset = GPUImageDataset(DATA_DIR, indices=train_idx, 
                                transform=base_transform, device=device)
val_dataset = GPUImageDataset(DATA_DIR, indices=val_idx, 
                              transform=base_transform, device=device)
test_dataset = GPUImageDataset(DATA_DIR, indices=test_idx, 
                               transform=base_transform, device=device)

train_counts = Counter(train_dataset.labels.cpu().numpy())
val_counts   = Counter(val_dataset.labels.cpu().numpy())
test_counts  = Counter(test_dataset.labels.cpu().numpy())

print(f'\nDataset statistics:')
print(f'Train: {len(train_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {train_counts[c]}')
print(f'Val:   {len(val_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {val_counts[c]}')
print(f'Test:  {len(test_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {test_counts[c]}')

## Class Weights

Inverse-frequency class weights are applied in the loss function to handle class imbalance.
No WeightedRandomSampler — experiments showed it hurts performance on this dataset.

In [ ]:
label_counts = Counter(train_dataset.labels.cpu().numpy())
total = len(train_dataset.labels)
class_weights = torch.zeros(NUM_CLASSES)
for i in range(NUM_CLASSES):
    class_weights[i] = total / (NUM_CLASSES * label_counts.get(i, 1))

print('Class weights (inverse frequency):')
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  {name}: weight={w:.4f}  (n={label_counts[i]})')
class_weights = class_weights.to(device)

print(f'\nAll class weights are now on {device}')

# Usando un modelo no preentrenado

# Usando un modelo preentrenado de EfficientNetB0 para clasificar la retinopatía diabética en cinco etapas (0-4).

## Model Architecture

**EfficientNetB0** pretrained on ImageNet, custom head: `Dropout(0.3) -> Linear(1280 -> 5)`

Best architecture for this dataset size — larger models overfit.

In [5]:
def build_model(num_classes=5, dropout=0.3, pretrained=True):
    """EfficientNetB0 with custom head for retinopathy classification."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Architecture: EfficientNetB0 (pretrained ImageNet)')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Classifier head:      Dropout(0.3) -> Linear(1280, {NUM_CLASSES})')

Architecture: EfficientNetB0 (pretrained ImageNet)
Total parameters:     4,013,953
Trainable parameters: 4,013,953
Classifier head:      Dropout(0.3) -> Linear(1280, 5)


## Training - Direct GPU Loading

Best hyperparameters:
- **Optimizer**: AdamW, lr=3e-4, weight_decay=1e-4
- **Scheduler**: OneCycleLR (max_lr=3e-4, pct_start=0.1, cosine annealing) — steps per batch
- **Loss**: CrossEntropyLoss + inverse-freq class weights + label_smoothing=0.1
- **Batch size**: 16, **Max epochs**: 200, **Early stopping patience**: 40
- **Gradient clipping**: max_norm=1.0
- **Image size**: 320x320 px
- **No augmentation or enhancement applied**

**Key optimization**: All images are pre-loaded directly onto the GPU. During training:
- ✅ No disk I/O (images cached in GPU memory)
- ✅ No CPU→GPU transfers (tensors already on GPU)
- ⚡ Result: Maximum training speed, limited only by GPU compute

In [ ]:
NUM_EPOCHS    = 200
BATCH_SIZE    = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 40
BEST_MODEL_PATH = os.path.join(model_dir, 'best_model_effb0_320px_cuda.pth')

# NOTE: pin_memory=False because data is already on GPU
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1, anneal_strategy='cos',
)
print(f'Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}')
print(f'Scheduler: OneCycleLR  max_lr={LEARNING_RATE}')
print(f'Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights')
print(f'\nNote: Images and labels are pre-loaded on {device}, no CPU→GPU transfer needed!')

In [ ]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on loader. Returns avg loss and accuracy."""
    model.eval()
    total_loss = 0.0
    preds, true_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            # Images and labels are already on device, no transfer needed
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(true_labels, preds)


best_val_acc     = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        # Images and labels are already on GPU, no .to(device) needed!
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0 or improved:
        marker = ' *' if improved else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS}  {avg_train_loss:>9.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}  {best_val_acc:>7.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed = time.time() - start_time
print(f'\nTraining finished in {elapsed/60:.1f} min')
print(f'Best Validation Accuracy: {best_val_acc:.4f}')

## Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        # Images already on GPU, no .to(device) needed
        outputs = model(images)
        all_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        all_true.extend(labels.cpu().numpy())

print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(all_true, all_preds))

## Second Model (for Ensemble)

Train a second EfficientNetB0 with a different random seed and earlier stopping point.
Ensembling two models trained on the same data but stopping at different times consistently
improves accuracy (+0.62% observed in R7+R8 ensemble on the full dataset).

In [ ]:
BEST_MODEL2_PATH = os.path.join(model_dir, 'best_model2_effb0_320px_cuda.pth')
NUM_EPOCHS_2 = 120
PATIENCE_2   = 30

torch.manual_seed(SEED + 1)
random.seed(SEED + 1)
np.random.seed(SEED + 1)

model2 = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

criterion2 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer2 = optim.AdamW(model2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler2 = optim.lr_scheduler.OneCycleLR(
    optimizer2, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS_2,
    pct_start=0.1, anneal_strategy='cos',
)

best_val_acc2     = 0.0
patience_counter2 = 0

print(f'Training second model (max {NUM_EPOCHS_2} epochs, patience {PATIENCE_2})...')
print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start2 = time.time()
for epoch in range(NUM_EPOCHS_2):
    model2.train()
    total_loss = 0.0
    for images, labels in train_loader:
        # Images and labels already on GPU
        optimizer2.zero_grad()
        loss = criterion2(model2(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
        optimizer2.step()
        scheduler2.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss2, val_acc2 = evaluate(model2, val_loader, criterion2, device)

    improved2 = val_acc2 > best_val_acc2
    if improved2:
        best_val_acc2 = val_acc2
        patience_counter2 = 0
        torch.save(model2.state_dict(), BEST_MODEL2_PATH)
    else:
        patience_counter2 += 1

    if (epoch + 1) % 10 == 0 or epoch == 0 or improved2:
        marker = ' *' if improved2 else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS_2}  {avg_train_loss:>9.4f}  {val_loss2:>8.4f}  {val_acc2:>7.4f}  {best_val_acc2:>7.4f}{marker}')

    if patience_counter2 >= PATIENCE_2:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed2 = time.time() - start2
print(f'\nTraining finished in {elapsed2/60:.1f} min')
print(f'Best Validation Accuracy (model2): {best_val_acc2:.4f}')

## Ensemble (Best Overall)

Average softmax probabilities of the two EfficientNetB0 models.
- Model 1: up to 200 epochs (best val checkpoint)
- Model 2: up to 120 epochs (best val checkpoint)

This strategy consistently yields the best test accuracy.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model2.load_state_dict(torch.load(BEST_MODEL2_PATH, map_location=device))
model.eval()
model2.eval()

ensemble_models = [model, model2]

# Individual model accuracies
_, acc1 = evaluate(model,  test_loader, criterion,  device)
_, acc2 = evaluate(model2, test_loader, criterion2, device)
print(f'Model 1 (200ep) Test Accuracy: {acc1:.4f}')
print(f'Model 2 (120ep) Test Accuracy: {acc2:.4f}')

# Ensemble: average softmax probabilities
ens_preds, ens_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        # Images already on GPU
        prob_sum = sum(F.softmax(m(images), dim=1) for m in ensemble_models)
        ens_preds.extend(torch.argmax(prob_sum, 1).cpu().numpy())
        ens_true.extend(labels.cpu().numpy())

ens_acc = accuracy_score(ens_true, ens_preds)
print(f'\nEnsemble Test Accuracy: {ens_acc:.4f}')
print('\nEnsemble Classification Report:')
print(classification_report(ens_true, ens_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(ens_true, ens_preds))

## Sample Inference

Test the ensemble on random samples from the test set.

In [ ]:
idx_to_class = {int(name.split('-')[0]): name for name in CLASS_NAMES}

print('Sample predictions (ensemble):')
print('-' * 65)
print(f"  {'Idx':>5}  {'True Label':>22}  {'Predicted':>22}  {'Correct':>7}")
print('-' * 65)

random.seed(0)
for _ in range(10):
    idx = random.randint(0, len(test_dataset) - 1)
    img, true_label = test_dataset[idx]
    with torch.no_grad():
        # Image already on GPU, just add batch dimension
        prob_sum = sum(F.softmax(m(img.unsqueeze(0)), dim=1) for m in ensemble_models)
        pred = torch.argmax(prob_sum, 1).item()
    correct = 'YES' if true_label.item() == pred else 'NO'
    print(f"  {idx:>5}  {idx_to_class[true_label.item()]:>22}  {idx_to_class[pred]:>22}  {correct:>7}")